# Evaluating Amazon Mistral Lite 7B finetuned for Skeptic Justifications dataset

### Install all dependencies

In [ ]:
# !pip install  accelerate --progress-bar off
# !pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
# !pip install  peft --progress-bar off
# !pip install  bitsandbytes --progress-bar off
# !pip install git+https://github.com/huggingface/transformers
# !pip install  xformers==0.0.21
# !pip install git+https://github.com/huggingface/trl.git
# !pip install deepspeed==0.9.5
# !pip install wandb
# !pip install vllm bert_score rouge nltk

### Loading Required Libraries

Next, we will load the required libraries for fine-tuning a Large Language Model (LLM)

In [1]:
import nltk
import torch
import random
import pandas as pd
import numpy as np
from rouge import Rouge
from bert_score import score
from vllm import LLM, SamplingParams
from nltk.translate.meteor_score import meteor_score

nltk.download('wordnet')
nltk.download('omw-1.4')

rouge = Rouge()

[nltk_data] Downloading package wordnet to /home/ec2-user/nltk_data...
[nltk_data] Downloading package omw-1.4 to /home/ec2-user/nltk_data...


In [2]:
torch_version = torch.__version__
if torch_version == "2.0.1+cu118":
    print(f"Torch version is satisfied: {torch.__version__}")
else:
    print("Torch version should be 2.0.1+cu118. Please ensure that before going further")

Torch version is satisfied: 2.0.1+cu118


### Loading the test set for Skeptic

In [3]:
df = pd.read_csv("data/test_skeptic_df.csv")

## vLLM Inference Server Engine for increased inference throughput and latency

In [4]:
# downloads finetuned model from huggingface hub
finetuned_model = "skshreyas714/skeptic-justify-205"

llm = LLM(model=finetuned_model, tensor_parallel_size=1)

INFO 10-29 06:18:16 llm_engine.py:72] Initializing an LLM engine with config: model='skshreyas714/skeptic-justify-205', tokenizer='skshreyas714/skeptic-justify-205', tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, quantization=None, seed=0)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


INFO 10-29 06:19:58 llm_engine.py:207] # GPU blocks: 1330, # CPU blocks: 2048


In [31]:
ids = [i for i in range(len(df))]
idx = random.choice(ids)
print(idx)

prompt = df.iloc[idx]["text"]
justify = df.iloc[idx]["Justification"]

print(f"Claim Summary: {prompt}")
print(f"Actual Output: {justify}")

16
Claim Summary: <|prompter|>You are a Financial Contrarian Writer. You are provided with a
Summary of Claims made by financial influencer and your task is to write
justifications for those claims. Claim Summary: The financial influencer claims that the share of Vodafone Idea has increased by more than 7.20% due to the Supreme Court's decision to review the appeal regarding Adjusted Gross Revenue (AGR). The influencer also mentions that Vodafone Idea and Bharti Airtel are disputing over the charges, claiming an error in the calculations made by the code implemented in 2019. Vodafone Idea has requested a reconsideration of the imposed penalties and interest, as they exceed the principal amount and could push the company into a severe financial crisis.</s><|assistant|>
Actual Output: The claim about the increase in Vodafone Idea's share due to the Supreme Court's decision to review the appeal regarding AGR is true as it is a common occurrence in the stock market for a company's share pr

In [6]:
test_prompts = df["text"].tolist()

In [34]:
sampling_params = SamplingParams(temperature=0.1, max_tokens=200, presence_penalty=1.5,
                                 frequency_penalty=1.5, top_p=0.9, top_k=50, best_of=10,
                                 skip_special_tokens=True, use_beam_search=False,
                                 early_stopping=False)

outputs = llm.generate(test_prompts, sampling_params)

predicted = []
for output in outputs:
    generated_text = output.outputs[0].text.strip(" ")
    predicted.append(generated_text)
    print(f"Generated text: {generated_text!r}")

Processed prompts: 100%|██████████| 21/21 [00:16<00:00,  1.24it/s]

Generated text: "The claims made by the influencer are true as they are based on the financial results of TCS for the second quarter of FY24. The increase in profits, revenues, and margins are indicators of a company's financial health. However, the decrease in dollar revenues is concerning as it suggests that TCS is losing business in foreign markets or facing currency headwinds. The announcement of a dividend and buyback shows that the company is confident about its future prospects and wants to reward its shareholders. However, whether this price jump can be sustained depends on various factors including market conditions and investor sentiment. Therefore, while these claims are factually accurate, they should be taken with caution from an investment perspective."
Generated text: "The claim that the conflict could impact India's economy is plausible as it could affect India's exports to Israel and West Asia. However, the specific impacts on premiums and shipping costs are harder to 

### Evaluation with ROUGE, METEOR, BERT-Score metrics

In [11]:
def compute_metrics(generated, reference):
    rouge_scores = rouge.get_scores(generated, reference)
    meteor = meteor_score([reference.split()], generated.split())
    bert_precision, bert_recall, bert_f1 = score([generated], [reference], lang="en")
    bert_f1 = bert_f1.detach().numpy().tolist()[0]
    return {"rouge_scores": rouge_scores, "meteor": meteor, "bert_score": bert_f1}

In [35]:
r, m, b = [], [], []
for i in range(len(df)):
    metrics = compute_metrics(predicted[i], df.iloc[i]["Justification"])
    rouge_m, meteor_m, bert_m = metrics["rouge_scores"][0]["rouge-l"]["f"], metrics["meteor"], metrics["bert_score"]
    r.append(rouge_m)
    m.append(meteor_m)
    b.append(bert_m)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['ro

In [36]:
rouge_s = np.round(np.mean(r), 2)
metoer_s = np.round(np.mean(m), 2)
bert_s = np.round(np.mean(b), 2)

In [37]:
print(f"Rouge Scores: {round(rouge_s,2)*100}%\n")
print(f"Meteor Scores: {round(metoer_s,2)*100}%\n")
print(f"BERT Scores: {round(bert_s,2)*100}%\n")

Rouge Scores: 54.0%

Meteor Scores: 43.0%

BERT Scores: 91.0%



### Dumping the generated justifications into test dataframe

In [19]:
df["Generated_Justifications"] = predicted

In [20]:
df.head()

,video_id,title,author,description,org_transcript,eng_transcript,image_base64,url,Summary_Claims,Justification,text,Generated_Justifications
0,7BObqdlgd5A,TCS Q2 Results 2023-24 Highlights | TCS Share ...,5paisa,TCS has just announced its Quarterly Results. ...,"Hi guys, Quarter 2 FY24 result season ki shirv...","""Hi guys, the Q2 FY24 result season has begun ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggIDQcICA...,https://www.youtube.com/watch?v=7BObqdlgd5A,The financial influencer claims that TCS's Q2 ...,The claims made by the influencer are true if ...,<|prompter|>You are a Financial Contrarian Wri...,The claims made by the influencer are true as ...
1,R_OryHP3Fcg,Israel-Hamas Conflict's Impact on India #shorts,5paisa,Gain insight into how the Israel-Hamas Conflic...,बिलियन डौलर का ट्रेटिंग रेलेशन्चिप खत्रे में ह...,The trading relationship between Israel and Ha...,/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=R_OryHP3Fcg,The financial influencer claims that the confl...,The claims made by the influencer are plausibl...,<|prompter|>You are a Financial Contrarian Wri...,The claim that the conflict could impact India...
2,qX3J9Tq0XYU,YOUTUBE SE INCOME || MY FIRST INCOME,Amrev,YOUTUBE SE INCOME || MY FIRST INCOME\r\n\r\n...,so yeah parents do say a lot but it's okay wh...,"""So yeah, parents do say a lot, but it's okay....",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAgICA...,https://www.youtube.com/watch?v=qX3J9Tq0XYU,No claims made,No claims to analyse,<|prompter|>You are a Financial Contrarian Wri...,No claims to analyse.
3,uQc6Tib329A,Growpital Review - 16% TAX FREE Return | Fixed...,Shrija Saha,Fixed Income - 16% Tax FREE Return - Growpita...,Fixed deposit with 16% returns वो भी tax-free ...,"""Fixed deposits with a 16% return are not tax-...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAUDBAgICAgICA...,https://www.youtube.com/watch?v=uQc6Tib329A\n,"The financial influencer claims that Gropetal,...",The claim of a 16% tax-free return in one year...,<|prompter|>You are a Financial Contrarian Wri...,The claim about a fixed return of 16% is hard ...
4,SQJxDy9F_7o,LOW RISK - HIGH REWARD || MY ADVICE FOR PEOPLE...,Amrev,LOW RISK - HIGH REWARD || MY ADVICE FOR PEOPLE...,सबसे पहले मेरा एक बहुत अफरेंट सबवाल है कि जो आ...,"First of all, my question is what suggestions ...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wCEAAgGBggICAcHBw...,https://www.youtube.com/watch?v=SQJxDy9F_7o,The financial influencer suggests that young i...,The claim that businesses with less risk have ...,<|prompter|>You are a Financial Contrarian Wri...,The claim that businesses with less risk have ...


In [21]:
df.to_csv("data/mistral-lite-finetuned-justifications.csv", index=False)